In [ ]:
import sys
sys.path.append('..')
import torch
from utils import DataHandler, Metric, train_unispecrec
from models import LightGCN

DEVICE = 'cuda'

datahandler = DataHandler(
    interaction_data='games',
    semantic_data='qwen',
    device=DEVICE,
    batch_size=4096,
    num_neg_item=64,
)
metric = Metric(device=DEVICE)

print(datahandler.get_info())

In [ ]:
model = LightGCN(
    rate_matrix=datahandler.rate_matrix,
    model_config=LightGCN.ModelConfig(
        latent_dim=32,
        num_layers=2,
        device=DEVICE,
        similarity='dot',
    ),
    loss_config=LightGCN.InfoNCELossConfig(
        type='infonce',
        similarity='cos',
        temperature=0.3,
        is_inbatch=True,
    ),
)

test_result = train_unispecrec(
    datahandler=datahandler,
    metric=metric,
    model=model,
    num_epochs=20000,
    num_steps=20,
    patience_steps=5,
    primary_metric='NDCG@20',
    verbose=False,
    use_amp=True,
    normalization_method='zscore',
)